# Horizon total return analysis

**Question this answers:** "If I hold this instrument for 3 months and spreads widen 25bp, what is my total return?"

Horizon total return composes carry, roll-down, and scenario-driven P&L
into a single return under user assumptions. It bridges the scenario engine
and P&L attribution framework: you supply a `ScenarioSpec` (which may include
a time-roll and market shocks), and the analyzer produces a factor-decomposed
total return.

## Setup

We build a 5-year USD corporate bond and price it against the shared demo
market (`_shared.build_demo_market()`), which supplies the `USD-OIS` discount
curve and the `CORP-HAZARD` credit curve the bond references. Horizon analysis
then runs under several scenarios.

In [1]:
import json

import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF, build_demo_market
from finstack_quant.scenarios import HorizonResult, compute_horizon_return


In [2]:
# 5-year 5% semi-annual corporate bond
bond_json = json.dumps({
    "type": "bond",
    "spec": {
        "id": "CORP-5Y-USD",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": DEMO_AS_OF.isoformat(),
        "maturity": "2030-01-15",
        "cashflow_spec": {
            "Fixed": {
                "coupon_type": "Cash",
                "rate": 0.05,
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "stub": "None",
                "end_of_month": False,
                "payment_lag_days": 0
            }
        },
        "discount_curve_id": "USD-OIS",
        "credit_curve_id": "CORP-HAZARD",
        "call_put": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {}
    }
})

print("Instrument ready: 5Y 5% USD corporate bond")

Instrument ready: 5Y 5% USD corporate bond


In [3]:
as_of = DEMO_AS_OF.isoformat()

# Shared cross-asset demo market: supplies the USD-OIS discount curve and the
# CORP-HAZARD credit curve referenced by the bond spec above.
mc = build_demo_market(DEMO_AS_OF)

print(f"Market ready (as_of {as_of}): USD-OIS discount + CORP-HAZARD credit curves")

Market ready (as_of 2025-01-15): USD-OIS discount + CORP-HAZARD credit curves


## Scenario 1: Pure carry

Hold the bond for 3 months, assuming markets don't move. All P&L comes from
coupon accrual, pull-to-par, and roll-down.

In [14]:
carry_scenario = json.dumps({
    "id": "carry-3m",
    "name": "3-Month Carry",
    "description": "Hold for 3 months, no market shocks",
    "operations": [
        {
            "kind": "time_roll_forward",
            "period": "3M",
            "apply_shocks": False,
            "roll_mode": "business_days"
        }
    ],
    "priority": 0,
    "resolution_mode": "most_specific_wins"
})

result = compute_horizon_return(bond_json, mc, as_of, carry_scenario)
print(result.explain())

Horizon Total Return: 2.2385%
Annualized: 9.3935%
Horizon: 90 days
Initial Value: USD 955846.05
Terminal Value: USD 977242.44
Total P&L: USD 21396.39
  Carry: USD 13466.13
  Rates: USD 0.00
  Credit: USD 7930.26
  Residual: USD 0.00



In [13]:
print(f"Horizon:          {result.horizon_days} days")
print(f"Total return:     {result.total_return_pct * 100:.4f}%")
print(f"Annualized:       {result.annualized_return * 100:.4f}%")
print(f"Carry component:  {result.factor_contribution('carry') * 100:.4f}%")

Horizon:          90 days
Total return:     2.2385%
Annualized:       9.3935%
Carry component:  1.4088%


## Scenario 2: Rate shock (no holding period)

What if rates rise 50bp right now? No time-roll, so carry is zero
and the result is a pure mark-to-scenario.

In [6]:
shock_scenario = json.dumps({
    "id": "rate-shock-50",
    "name": "Rates +50bp",
    "description": "Instantaneous parallel rate shock",
    "operations": [
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "bp": 50.0
        }
    ],
    "priority": 0,
    "resolution_mode": "most_specific_wins"
})

result = compute_horizon_return(bond_json, mc, as_of, shock_scenario)
print(result.explain())
print(f"Horizon days:     {result.horizon_days}  (None = no time-roll)")
print(f"Annualized:       {result.annualized_return}  (None = not meaningful)")

Horizon Total Return: -2.1612%
Initial Value: USD 955846.05
Terminal Value: USD 935188.24
Total P&L: USD -20657.81
  Carry: USD 0.00
  Rates: USD -20657.81
  Credit: USD 0.00
  Residual: USD 0.00

Horizon days:     None  (None = no time-roll)
Annualized:       None  (None = not meaningful)


## Scenario 3: Hold 3 months + spreads widen 25bp

The core use case. Combines a holding period with a market view:
earn carry while credit deteriorates.

In [7]:
combined_scenario = json.dumps({
    "id": "hold-3m-spread-25",
    "name": "3M Hold + Spread Widening",
    "description": "Hold for 3 months while credit spreads widen 25bp",
    "operations": [
        {
            "kind": "time_roll_forward",
            "period": "3M",
            "apply_shocks": True,
            "roll_mode": "business_days"
        },
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "par_cds",
            "curve_id": "CORP-HAZARD",
            "discount_curve_id": "USD-OIS",
            "bp": 25.0
        }
    ],
    "priority": 0,
    "resolution_mode": "most_specific_wins"
})

result = compute_horizon_return(bond_json, mc, as_of, combined_scenario)
print(result.explain())

Horizon Total Return: 0.3958%
Annualized: 1.6151%
Horizon: 90 days
Initial Value: USD 955846.05
Terminal Value: USD 959629.61
Total P&L: USD 3783.56
  Carry: USD 13466.13
  Rates: USD 0.00
  Credit: USD -9682.57
  Residual: USD 0.00



In [8]:
# Factor contributions as percentage of initial value
factors = ["carry", "rates", "credit", "fx", "vol"]
print("Factor contributions:")
for f in factors:
    pct = result.factor_contribution(f) * 100
    print(f"  {f:12s}  {pct:+.4f}%")

print(f"\n  {'total':12s}  {result.total_return_pct * 100:+.4f}%")
print(f"  annualized:   {result.annualized_return * 100:+.4f}%")

Factor contributions:
  carry         +1.4088%
  rates         +0.0000%
  credit        -1.0130%
  fx            +0.0000%
  vol           +0.0000%

  total         +0.3958%
  annualized:   +1.6151%


## Scenario 4: Multi-factor stress

Hold for 3 months with rates up 100bp and spreads widening 50bp.
A more adverse scenario combining rate and credit moves.

In [9]:
stress_scenario = json.dumps({
    "id": "hold-3m-stress",
    "name": "3M Hold Under Stress",
    "description": "Hold 3 months, rates +100bp, spreads +50bp",
    "operations": [
        {
            "kind": "time_roll_forward",
            "period": "3M",
            "apply_shocks": True,
            "roll_mode": "business_days"
        },
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "bp": 100.0
        },
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "par_cds",
            "curve_id": "CORP-HAZARD",
            "discount_curve_id": "USD-OIS",
            "bp": 50.0
        }
    ],
    "priority": 0,
    "resolution_mode": "most_specific_wins"
})

result = compute_horizon_return(bond_json, mc, as_of, stress_scenario)
print(result.explain())

Horizon Total Return: -4.6109%
Annualized: -17.4236%
Horizon: 90 days
Initial Value: USD 955846.05
Terminal Value: USD 911772.81
Total P&L: USD -44073.24
  Carry: USD 13466.13
  Rates: USD -38048.14
  Credit: USD -18341.50
  Residual: USD 0.00



## Attribution methods

The `method` parameter controls how P&L is decomposed:
- `"parallel"` (default) -- isolate each factor independently
- `"waterfall"` -- sequential application, sum = total by construction
- `"metrics_based"` -- fast linear approximation via DV01/CS01
- `"taylor"` -- sensitivity-based Taylor expansion

In [10]:
for method in ["parallel", "waterfall"]:
    r = compute_horizon_return(bond_json, mc, as_of, combined_scenario, method=method)
    print(f"--- {method} ---")
    print(f"  Total return:  {r.total_return_pct * 100:+.4f}%")
    print(f"  Carry:         {r.attribution.carry:.2f}")
    print(f"  Credit:        {r.attribution.credit_curves_pnl:.2f}")
    print(f"  Residual:      {r.attribution.residual:.2f}")
    print()

--- parallel ---
  Total return:  +0.3958%
  Carry:         13466.13
  Credit:        -9682.57
  Residual:      0.00

--- waterfall ---
  Total return:  +0.3958%
  Carry:         13466.13
  Credit:        -9682.57
  Residual:      0.00



## Serialization

Results can be serialized to JSON for storage or downstream processing.
The full `PnlAttribution` is available via `.attribution`.

In [11]:
result = compute_horizon_return(bond_json, mc, as_of, combined_scenario)

# Access the full PnlAttribution object
attr = result.attribution
print(f"Attribution method: {attr.method}")
print(f"T0: {attr.t0}  ->  T1: {attr.t1}")
print(f"Residual %: {attr.residual_pct:.4f}%")

# Serialize entire result to JSON
result_json = result.to_json()
print(f"\nJSON output size: {len(result_json)} chars")
print(json.loads(result_json).keys())

Attribution method: Parallel
T0: 2025-01-15  ->  T1: 2025-04-15
Residual %: 0.0000%

JSON output size: 3444 chars
dict_keys(['attribution', 'initial_value', 'terminal_value', 'horizon_days', 'scenario_report'])


## Takeaways

- `compute_horizon_return` composes any `ScenarioSpec` with P&L attribution
- Include a `time_roll_forward` operation for a holding period; omit it for pure mark-to-scenario
- The result wraps a full `PnlAttribution` with convenience accessors for total return, annualized return, and per-factor contributions
- Four attribution methods are available; `"parallel"` is the default